In [1]:
import pandas as pd

from Pipeline.Global.GallstoneDataSet import GallstoneDataSet
from Pipeline.Methodology.EvaluationELM import EvaluationELM
from Pipeline.Global.GlobalSetting import GlobalSetting

In [2]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_cleaned_data_path()
gallstone_dataset.normal_data_split()

features_size = gallstone_dataset.x_train.shape[1]

In [3]:
# 1. Instantiate the Evaluator Object ONCE
evaluator = EvaluationELM(
    x_train = gallstone_dataset.x_train,
    y_train = gallstone_dataset.y_train,
    activation_function     = GlobalSetting.sigmoid ,
    elm_init_seed_range     = GlobalSetting.elm_initial_state_range ,
    k_fold = 5
)

In [4]:
hidden_size_record ,hidden_size_result = evaluator.grid_search_hidden_size(
    GlobalSetting.hidden_size_explore_range
)
GlobalSetting.save_dataframe_to_record(hidden_size_record,'hidden_size_record.csv')
hidden_node_list = EvaluationELM.extract_top_results(hidden_size_result)
best_hidden_size = hidden_node_list.iloc[0]

[I/O Trace] Record exported successfully: C:\Users\yqn1e23\PycharmProjects\COMP3200_Individual_Project\Storage\Record\hidden_size_record.csv


In [5]:
lambda_record, lambda_result = evaluator.grid_search_lambda(
    hidden_size     = best_hidden_size['Hidden_Nodes'],
    lambda_range    = GlobalSetting.lambda_explore_range
)
GlobalSetting.save_dataframe_to_record(lambda_record,'lambda_record.csv')
lambda_value_list = EvaluationELM.extract_top_results(lambda_result)
best_lambda_value = lambda_value_list.iloc[0]

[I/O Trace] Record exported successfully: C:\Users\yqn1e23\PycharmProjects\COMP3200_Individual_Project\Storage\Record\lambda_record.csv


In [6]:
size_lambda_record , size_lambda_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range   = hidden_node_list['Hidden_Nodes'],
    lambda_range        = GlobalSetting.lambda_explore_range
)
GlobalSetting.save_dataframe_to_record(size_lambda_record,'size_lambda_record.csv')
size_lambda_list = EvaluationELM.extract_top_results(size_lambda_result)
best_size_lambda = size_lambda_list.iloc[0]

[I/O Trace] Record exported successfully: C:\Users\yqn1e23\PycharmProjects\COMP3200_Individual_Project\Storage\Record\size_lambda_record.csv


In [7]:
combination_record , combination_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range   = GlobalSetting.hidden_size_explore_range,
    lambda_range        = GlobalSetting.lambda_explore_range
)
GlobalSetting.save_dataframe_to_record(combination_record,'combination_record')
combination_result_list = EvaluationELM.extract_top_results(combination_result)
best_combination_result = combination_result_list.iloc[0]

[I/O Trace] Record exported successfully: C:\Users\yqn1e23\PycharmProjects\COMP3200_Individual_Project\Storage\Record\combination_record.csv


In [8]:
hidden_size_smaller = range(10, round(best_hidden_size['Hidden_Nodes'] * 0.8),5)
optimization_record , optimization_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range = hidden_size_smaller,
    lambda_range = GlobalSetting.lambda_search_range
)
GlobalSetting.save_dataframe_to_record(optimization_record,'optimization_record.csv')
optimization_result_list = EvaluationELM.extract_top_results(optimization_result)
best_optimization_result = optimization_result_list.iloc[0]

[I/O Trace] Record exported successfully: C:\Users\yqn1e23\PycharmProjects\COMP3200_Individual_Project\Storage\Record\optimization_record.csv


In [9]:
optimization_hidden_record , optimization_hidden_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range = [best_optimization_result['Hidden_Nodes']],
    lambda_range = [0.0]
)
GlobalSetting.save_dataframe_to_record(optimization_hidden_record,'optimization_hidden_record.csv')
optimization_hidden_result_list = EvaluationELM.extract_top_results(optimization_hidden_result)
optimization_hidden_result = optimization_hidden_result_list.iloc[0]

[I/O Trace] Record exported successfully: C:\Users\yqn1e23\PycharmProjects\COMP3200_Individual_Project\Storage\Record\optimization_hidden_record.csv


In [10]:
model_configs_payload = [
    {
        "Model_Types"   : "Best_Hidden_Nodes",
        "Hidden_Nodes"  : int(best_hidden_size['Hidden_Nodes']),
        "Activation"    : "sigmoid",
        "Lambda_Value"  : float(best_hidden_size['Lambda_Value'])
    },
    {
        "Model_Types"   : "Best_Lambda",
        "Hidden_Nodes"  : int(best_lambda_value['Hidden_Nodes']),
        "Activation"    : "sigmoid",
        "Lambda_Value"  : float(best_lambda_value['Lambda_Value'])
    },
    {
        "Model_Types"   : "Best_Size_and_Lambda",
        "Hidden_Nodes"  : int(best_size_lambda['Hidden_Nodes']),
        "Activation"    : "sigmoid",
        "Lambda_Value"  : float(best_size_lambda['Lambda_Value'])
    },
    {
        "Model_Types"   : "Grid_Combination",
        "Hidden_Nodes"  : int(best_combination_result['Hidden_Nodes']),
        "Activation"    : "sigmoid",
        "Lambda_Value"  : float(best_combination_result['Lambda_Value'])
    },
    {
        "Model_Types"   : "Grid_Optimization",
        "Hidden_Nodes"  : int(best_optimization_result['Hidden_Nodes']),
        "Activation"    : "sigmoid",
        "Lambda_Value"  : float(best_optimization_result['Lambda_Value'])
    },
    {
        "Model_Types"   : "Optimization_Hidden",
        "Hidden_Nodes"  : int(optimization_hidden_result['Hidden_Nodes']),
        "Activation"    : "sigmoid",
        "Lambda_Value"  : float(optimization_hidden_result['Lambda_Value'])
    }
]
GlobalSetting.upsert_model_configs(model_configs_payload)

In [11]:
combined_df = pd.concat([best_hidden_size, best_lambda_value, best_size_lambda,best_combination_result, best_optimization_result, optimization_hidden_result], axis=1)
combined_df

,8,23,23,528,319,0
Hidden_Nodes,50,50,50,100,35,35
Activation,sigmoid,sigmoid,sigmoid,sigmoid,sigmoid,sigmoid
Lambda_Value,0.0,0.25,0.25,0.5,0.125,0.0
lcb_Accuracy_Seed_Mean,0.725304,0.741254,0.741254,0.742113,0.729974,0.711057
lcb_Accuracy_Seed_Std,0.02733,0.021567,0.021567,0.014653,0.029361,0.02783
lcb_Accuracy_Seed_SEM,0.00499,0.003938,0.003938,0.002675,0.005361,0.005081
lcb_Accuracy_Seed_Min,0.658508,0.689838,0.689838,0.711178,0.658337,0.671782
lcb_Accuracy_Seed_Max,0.767687,0.786194,0.786194,0.76819,0.778111,0.77176
lcb_Precision_Seed_Mean,0.732054,0.74569,0.74569,0.752457,0.73181,0.717262
lcb_Precision_Seed_Std,0.031206,0.035212,0.035212,0.021506,0.035924,0.033551
